# MOSS-Image LoRA на Colab TPU (v5e-1)

Порт обучения LoRA из kohya-ss/sd-scripts на JAX/TPU: [moss-image-jax-tpu](https://github.com/drimeF0/moss-image-jax-tpu) (план — `PLAN.md`).

- Цель: Colab TPU v5e-1 (1 чип, 16 ГБ HBM), база в int8, полный пайплайн на TPU.
- **Правило одного процесса**: всё, что инициализирует JAX/TPU, запускается только через `!python scripts/colab_run.py ...` в одном процессе. Не вызывайте `jax.devices()` или jnp-операции в ячейках ноутбука — иначе TPU останется занят и раннер упадёт.
- Порядок: smoke-тест (без скачиваний) → реальный прогон.

In [ ]:
# Информация о среде (JAX не инициализируется)
!nvidia-smi -L 2>/dev/null || true
!ls /dev/accel* 2>/dev/null || echo 'no /dev/accel*'
!pip show jaxlib 2>/dev/null | head -3
!df -h /content | tail -1

In [ ]:
# Клон репозитория и установка зависимостей
%cd /content
!git clone -q https://github.com/drimeF0/moss-image-jax-tpu.git
%cd /content/moss-image-jax-tpu
!pip install -q -e .
!pip install -q optax safetensors transformers pillow huggingface_hub
# jax/jaxlib уже стоят в TPU-рантайме Colab — не переустанавливайте их.

## 1. Smoke-тест (tiny-модели, без скачивания весов)

Проверяет весь контур: случайные tiny VAE/TE/DiT → кэши → обучение → сэмплинг. Занимает 2-5 минут на TPU.

In [ ]:
!cd /content/moss-image-jax-tpu && python scripts/colab_run.py --mode smoke --output-dir /content/moss_out/smoke

In [ ]:
import json
from IPython.display import Image, display

summary = json.load(open('/content/moss_out/smoke/summary.json'))
print(json.dumps(summary, indent=2))
display(Image('/content/moss_out/smoke/smoke_sample.png'))

## 2. Реальный прогон

Нужны: DiT (официальные shards), Qwen3.5-9B TE, FLUX.2 VAE, tokenizer. Всё есть в репозитории `OpenMOSS-Team/MOSS-Image-Base` (~35 ГБ; при нехватке RAM выберите high-RAM runtime — квантование в int8 идёт на хосте).

Данные: `images_dir` с картинками и `prompts.json` вида `{"file.png": ["caption 1", "caption 2"]}` (кандидаты капшенов поддерживаются).

In [ ]:
# Скачивание моделей (можно пропустить, если веса уже на диске)
!pip install -q 'huggingface_hub[hf_transfer]'
!HF_HUB_ENABLE_HF_TRANSFER=1 python -c "from huggingface_hub import snapshot_download; snapshot_download('OpenMOSS-Team/MOSS-Image-Base', local_dir='/content/models/moss-image-base', allow_patterns=['transformer/*', 'text_encoder/*', 'vae/*', 'tokenizer/*'])"
!ls /content/models/moss-image-base && ls /content/models/moss-image-base/transformer | head -5

### 2b. Пре-квантизация (один раз)

Сохраняет int8-веса DiT и TE (плюс fp32-скейлы) в один safetensors каждый. Дальше загрузка — простое чтение файла без 20-минутного квантования; `--int8` для этих файлов не нужен (лоадер распознаёт их автоматически).

In [ ]:
!cd /content/moss-image-jax-tpu && python scripts/prepare_quantized.py \
  --dit /content/models/moss-image-base/transformer/diffusion_pytorch_model-00001-of-00003.safetensors \
  --text-encoder /content/models/moss-image-base/text_encoder/model-00001-of-00004.safetensors \
  --output-dir /content/models/prepared
!cat /content/models/prepared/prepare_summary.json

In [ ]:
# Минимальный датасет для проверки (замените на свои картинки и капшены)
import json, os
import numpy as np
from PIL import Image

os.makedirs('/content/data/images', exist_ok=True)
image_path = '/content/data/images/example.png'
if not os.path.exists(image_path):
    Image.fromarray((np.random.rand(768, 768, 3) * 255).astype('uint8')).save(image_path)
json.dump({os.path.basename(image_path): ['a test image for the pipeline run']}, open('/content/data/prompts.json', 'w'))
print('dataset ready')

In [ ]:
# Полный прогон в одном TPU-процессе: кэши -> обучение -> сэмпл
# Используются пре-квантизованные файлы из шага 2b; --int8 больше не нужен.
# Для первого запуска: --max-samples 1 --steps 20; далее увеличьте шаги.
!cd /content/moss-image-jax-tpu && python scripts/colab_run.py --mode full \
  --dit /content/models/prepared/moss_image_base_dit_int8.safetensors \
  --vae /content/models/moss-image-base/vae/diffusion_pytorch_model.safetensors \
  --text-encoder /content/models/prepared/moss_image_base_te_int8.safetensors \
  --tokenizer /content/models/moss-image-base/tokenizer \
  --images-dir /content/data/images --prompts /content/data/prompts.json \
  --output-dir /content/moss_out/run1 --rank 16 --steps 20 --max-samples 1 \
  --width 512 --height 512

In [ ]:
import json, os
from IPython.display import Image, display

summary = json.load(open('/content/moss_out/run1/summary.json'))
print(json.dumps(summary, indent=2))
if os.path.exists('/content/moss_out/run1/sample.png'):
    display(Image('/content/moss_out/run1/sample.png'))
print('LoRA:', '/content/moss_out/run1/checkpoints/moss_lora_final.safetensors')
# Сохранение на Google Drive:
# from google.colab import drive; drive.mount('/content/drive')
# !cp /content/moss_out/run1/checkpoints/moss_lora_final.safetensors /content/drive/MyDrive/

## Заметки

- **Один TPU-процесс за сессию**: если прогон упал или прерван, перезапустите runtime (Runtime → Restart) перед следующим запуском.
- 16 ГБ HBM: TE (int8 ~9 ГБ) и DiT (int8 ~5 ГБ) загружаются по очереди и освобождаются между стадиями; не запускайте кэширование и обучение параллельно.
- При нехватке памяти: уменьшите `--width/--height`, число шагов, добавьте `--skip-sample`; для TE/VAE-стадий используйте high-RAM runtime.
- Итоги стадий и ошибки: `summary.json` в `--output-dir`; LoRA-файл совместим с upstream `networks.lora_generic` (проверено roundtrip-тестом).
- Скрипты без ноутбука: `make_caches.py`, `train.py`, `sample.py`; датасет можно задать `.toml` через `--dataset-config`.